# Angular Interview Q&A — Set 1
### Core Fundamentals | Instructor Reference Guide

---

## About This Set

This is **Set 1** of the Angular Interview Q&A series, covering **core Angular fundamentals** that every Angular developer — from junior to senior — must know.

| Category | Questions | Topics |
|---|---|---|
| **1. Architecture & Fundamentals** | Q1 – Q5 | Angular overview, NgModule, Standalone, CLI, build |
| **2. Components & Templates** | Q6 – Q10 | Lifecycle hooks, Input/Output, ViewChild, template syntax |
| **3. Directives, Pipes & Data Binding** | Q11 – Q15 | Structural vs Attribute, custom pipes, binding types |
| **4. Services, DI & HTTP** | Q16 – Q20 | Dependency injection, HttpClient, interceptors, RxJS |
| **5. Routing & Navigation** | Q21 – Q25 | RouterModule, lazy loading, guards, resolvers |
| **6. Forms** | Q26 – Q30 | Template-driven vs Reactive, validation, FormBuilder |

**Total Questions: 30**

---

> **How to use this guide:**
> - Questions are ordered from foundational → advanced within each category
> - Each answer includes a code example where applicable
> - Look for **"Follow-up:"** sections — interviewers commonly ask these next
> - Look for **"Trap:"** sections — common mistakes candidates make

# Category 1 — Architecture & Fundamentals

---

## Q1. What is Angular? How is it different from AngularJS?

**Answer:**
Angular is a **TypeScript-based open-source front-end web framework** developed and maintained by Google. It is used to build Single Page Applications (SPAs) with a component-based architecture.

| Aspect | AngularJS (v1.x) | Angular (v2+) |
|---|---|---|
| Language | JavaScript | TypeScript |
| Architecture | MVC (Model-View-Controller) | Component-based |
| Data Binding | Two-way (scope-based) | One-way by default + two-way via `[(ngModel)]` |
| Mobile Support | Limited | Designed for mobile |
| Performance | Slower (dirty checking all scope) | Faster (OnPush, Signals, tree-shaking) |
| CLI | Minimal tooling | Angular CLI (powerful) |
| Dependency Injection | Basic | Hierarchical DI system |
| Release Cycle | No fixed cadence | Major release every 6 months |

**Key Angular architecture layers:**
```
Browser
  └── Angular Application
        ├── Modules (or Standalone Components)
        │     ├── Components (HTML + TypeScript + CSS)
        │     ├── Directives
        │     ├── Pipes
        │     └── Services (via DI)
        ├── Router
        ├── HttpClient
        └── RxJS / Signals
```

> **Follow-up:** "What is the current Angular version?" → Angular 18 (May 2024), Angular 19 (Nov 2024), Angular 20 (May 2025)

---

## Q2. What is `NgModule`? What are its key properties?

**Answer:**
`NgModule` is a **decorator that groups related Angular building blocks** — components, directives, pipes, and services — into a cohesive unit. Before Angular 14, every Angular app required at least one `NgModule` (the root `AppModule`).

```typescript
import { NgModule } from '@angular/core';
import { BrowserModule } from '@angular/platform-browser';
import { HttpClientModule } from '@angular/common/http';
import { AppRoutingModule } from './app-routing.module';
import { AppComponent } from './app.component';
import { UserComponent } from './user/user.component';
import { UserService } from './user/user.service';

@NgModule({
  declarations: [
    AppComponent,        // Components, Directives, Pipes owned by this module
    UserComponent,
  ],
  imports: [
    BrowserModule,       // Other modules this module depends on
    HttpClientModule,
    AppRoutingModule,
  ],
  providers: [
    UserService,         // Services (now prefer providedIn: 'root')
  ],
  exports: [
    UserComponent,       // Make available to other modules that import this module
  ],
  bootstrap: [AppComponent]  // Root component to bootstrap (AppModule only)
})
export class AppModule {}
```

**Key properties summary:**

| Property | Purpose |
|---|---|
| `declarations` | Components, Directives, Pipes belonging to this module |
| `imports` | Other modules needed by this module |
| `providers` | Services registered in this module's injector |
| `exports` | Publicly expose declarations to other modules |
| `bootstrap` | The root component (only in AppModule) |

> **Follow-up:** "What replaced NgModules?" → Standalone Components (stable since Angular 15/17)

> **Trap:** You cannot declare the same component in two different `NgModule.declarations`. It will throw a runtime error.

---

## Q3. What are Standalone Components? How do they differ from NgModule-based components?

**Answer:**
A **Standalone Component** is a component that manages its own dependencies directly — without being declared in an `NgModule`. Introduced in Angular 14 (Developer Preview) and stable since Angular 15. As of Angular 19, `standalone: true` is the **default**.

```typescript
// NgModule-based (traditional)
@Component({
  selector: 'app-user',
  template: `<p>{{ user.name }}</p>`,
})
export class UserComponent {}  // Must be declared in NgModule.declarations

// ------------------------------------------

// Standalone (modern)
@Component({
  selector: 'app-user',
  standalone: true,
  imports: [CommonModule, RouterLink, AsyncPipe],  // Direct imports
  template: `<p>{{ user.name }}</p>`,
})
export class UserComponent {}  // No NgModule needed
```

**Bootstrapping a standalone app:**
```typescript
// main.ts
import { bootstrapApplication } from '@angular/platform-browser';
import { provideRouter } from '@angular/router';
import { provideHttpClient } from '@angular/common/http';
import { AppComponent } from './app/app.component';

bootstrapApplication(AppComponent, {
  providers: [
    provideRouter(routes),
    provideHttpClient(),
  ]
});
```

**Comparison:**

| Feature | NgModule | Standalone |
|---|---|---|
| `declarations` array | Required | ❌ Not used |
| Imports in component | Via NgModule | Direct in `@Component.imports` |
| Lazy loading | Module-level | Component-level |
| Boilerplate | More | Less |
| Default (Angular 19+) | No | ✅ Yes |

> **Trap:** In a standalone component, you must import `CommonModule` (or specific directives like `NgIf`, `NgFor`) directly, not rely on a parent module to provide them.

---

## Q4. What is the Angular CLI? List the most important CLI commands.

**Answer:**
The **Angular CLI (Command Line Interface)** is a toolchain for initializing, developing, scaffolding, and maintaining Angular applications. It abstracts build tools (Webpack, Vite, esbuild) and enforces Angular best practices.

```bash
# Install CLI globally
npm install -g @angular/cli

# Create a new project
ng new my-app --standalone --routing --style=scss

# Start development server
ng serve --open                    # Opens browser automatically
ng serve --port 4200 --hmr         # Custom port + Hot Module Replacement

# Generate code
ng generate component user/profile  # or: ng g c user/profile
ng generate service core/auth       # or: ng g s core/auth
ng generate module features/admin --routing
ng generate pipe shared/truncate
ng generate directive shared/highlight
ng generate guard core/auth --implements CanActivate
ng generate interface models/user

# Build
ng build                           # Development build
ng build --configuration production  # Production build (optimized)

# Testing
ng test                            # Unit tests (Karma + Jasmine)
ng e2e                             # End-to-end tests

# Upgrade
ng update @angular/core @angular/cli  # Update Angular

# Lint
ng lint

# Analyze bundle
ng build --stats-json
npx webpack-bundle-analyzer dist/my-app/stats.json
```

**Important CLI flags:**

| Flag | Description |
|---|---|
| `--dry-run` or `-d` | Preview what will be generated without creating files |
| `--skip-tests` | Skip spec file generation |
| `--inline-template` | Put template in the component TS file |
| `--prefix` | Set selector prefix (default: `app`) |

---

## Q5. What is the Angular build process? What happens when you run `ng build`?

**Answer:**
`ng build` compiles and bundles the Angular application for deployment. Since Angular 17, the default builder is **`application` (Vite + esbuild)**, replacing the older Webpack-based builder.

**Build pipeline steps:**
```
Source Code (TypeScript + HTML + CSS/SCSS)
    │
    ▼
1. TypeScript Compilation (tsc)
   └── Type-checking, transpilation to ES2022+
    │
    ▼
2. Angular Compilation (ngc / Ivy compiler)
   └── Compiles templates → JavaScript (no template interpreter at runtime)
   └── Tree-shaking: removes unused Angular code
    │
    ▼
3. Bundling (esbuild in Angular 17+)
   └── Creates JS bundles: main, polyfills, runtime, lazy chunks
    │
    ▼
4. Optimization
   └── Minification, dead code elimination
   └── CSS extraction and optimization
   └── Source map generation (optional)
    │
    ▼
5. Output: dist/ folder
   └── index.html
   └── main.[hash].js
   └── polyfills.[hash].js
   └── [chunk].[hash].js  (lazy-loaded routes)
   └── styles.[hash].css
   └── assets/
```

**Output files explained:**

| File | Purpose |
|---|---|
| `main.[hash].js` | App code — components, services, etc. |
| `polyfills.[hash].js` | Browser compatibility polyfills |
| `runtime.[hash].js` | Angular module loader |
| `[chunk].[hash].js` | Lazy-loaded route/module bundles |

**Development vs Production:**

| Aspect | `ng build` (dev) | `ng build --configuration production` |
|---|---|---|
| Source maps | Full | Hidden/none |
| Minification | No | Yes |
| Tree-shaking | Partial | Aggressive |
| AOT Compilation | Yes (v9+) | Yes |
| Bundle size | Larger | Smaller |

> **Follow-up:** "What is AOT vs JIT compilation?"
> - **AOT (Ahead-of-Time):** Templates compiled at build time → faster startup, smaller bundle, catches template errors early. Default since Angular 9.
> - **JIT (Just-in-Time):** Templates compiled in the browser at runtime → used in older versions, slower startup.

# Category 2 — Components & Templates

---

## Q6. What are Angular Component Lifecycle Hooks? List them in order.

**Answer:**
Lifecycle hooks are **interface methods** that Angular calls at specific points during a component's life — from creation to destruction. They let you tap into key moments to run custom logic.

**Lifecycle order:**

| # | Hook | Interface | When Called |
|---|---|---|---|
| 1 | `ngOnChanges` | `OnChanges` | Before `ngOnInit`, and every time an `@Input` value changes |
| 2 | `ngOnInit` | `OnInit` | Once, after first `ngOnChanges` — component initialized |
| 3 | `ngDoCheck` | `DoCheck` | Every change detection cycle (use carefully — fires often) |
| 4 | `ngAfterContentInit` | `AfterContentInit` | Once, after `ng-content` is projected into the component |
| 5 | `ngAfterContentChecked` | `AfterContentChecked` | After every check of projected content |
| 6 | `ngAfterViewInit` | `AfterViewInit` | Once, after component's own view (and children) is initialized |
| 7 | `ngAfterViewChecked` | `AfterViewChecked` | After every check of the component's view |
| 8 | `ngOnDestroy` | `OnDestroy` | Just before the component is destroyed |

```typescript
import {
  Component, Input, OnChanges, OnInit, DoCheck,
  AfterContentInit, AfterContentChecked,
  AfterViewInit, AfterViewChecked, OnDestroy,
  SimpleChanges
} from '@angular/core';

@Component({
  selector: 'app-lifecycle-demo',
  standalone: true,
  template: `<p>Count: {{ count }}</p>`
})
export class LifecycleDemoComponent
  implements OnChanges, OnInit, DoCheck, AfterContentInit,
             AfterContentChecked, AfterViewInit, AfterViewChecked, OnDestroy {

  @Input() count = 0;

  ngOnChanges(changes: SimpleChanges) {
    console.log('1. ngOnChanges', changes);
    // changes['count'].currentValue, previousValue, firstChange
  }

  ngOnInit() {
    console.log('2. ngOnInit — safe to call APIs here');
  }

  ngDoCheck() {
    console.log('3. ngDoCheck — runs every CD cycle');
  }

  ngAfterContentInit()    { console.log('4. ngAfterContentInit'); }
  ngAfterContentChecked() { console.log('5. ngAfterContentChecked'); }
  ngAfterViewInit()       { console.log('6. ngAfterViewInit — safe to access ViewChild here'); }
  ngAfterViewChecked()    { console.log('7. ngAfterViewChecked'); }

  ngOnDestroy() {
    console.log('8. ngOnDestroy — unsubscribe here!');
  }
}
```

**Most commonly used hooks:**
- **`ngOnInit`** — Fetch data, initialize subscriptions
- **`ngOnChanges`** — React to `@Input` changes from parent
- **`ngOnDestroy`** — Unsubscribe from Observables, clear timers
- **`ngAfterViewInit`** — Access `@ViewChild` elements (DOM is ready)

> **Trap:** Never access `@ViewChild` in `ngOnInit` — the view isn't rendered yet. Use `ngAfterViewInit` instead.

---

## Q7. What is the difference between `@Input()` and `@Output()`? How does component communication work?

**Answer:**
Angular components communicate through **property binding** (`@Input`) for parent→child data flow, and **event binding** (`@Output`) for child→parent communication.

```typescript
// child.component.ts
import { Component, Input, Output, EventEmitter } from '@angular/core';

@Component({
  selector: 'app-product-card',
  standalone: true,
  template: `
    <div class="card">
      <h3>{{ title }}</h3>
      <p>{{ price | currency }}</p>
      <button (click)="onAddToCart()">Add to Cart</button>
    </div>
  `
})
export class ProductCardComponent {
  @Input() title: string = '';         // Receives data from parent
  @Input() price: number = 0;
  @Input({ required: true }) productId!: string;  // Required since Angular 16

  @Output() addedToCart = new EventEmitter<string>();  // Sends event to parent

  onAddToCart() {
    this.addedToCart.emit(this.productId);
  }
}
```

```html
<!-- parent.component.html -->
<app-product-card
  title="Laptop"
  [price]="product.price"
  [productId]="product.id"
  (addedToCart)="handleAddToCart($event)">
</app-product-card>
```

```typescript
// parent.component.ts
handleAddToCart(productId: string) {
  this.cartService.add(productId);
}
```

**Angular 14+ — Typed `@Input()`:**
```typescript
@Input() items: string[] = [];         // TypeScript typed
@Input({ alias: 'itemTitle' }) title!: string;  // Alias
```

**Modern Angular 16+ — Signal inputs:**
```typescript
import { input, output } from '@angular/core';

title    = input.required<string>();    // Required signal input
price    = input(0);                    // Optional with default
addedToCart = output<string>();        // Signal output
```

> **Follow-up:** "What are other ways components can communicate?"
> - **Sibling → Sibling:** Use a shared Service with `Subject` / `BehaviorSubject` / `signal()`
> - **Any component:** Use Angular Signals Store, NgRx, or a root-level service

---

## Q8. What is `@ViewChild` and `@ContentChild`? When do you use each?

**Answer:**

**`@ViewChild`** — Access a **child component, directive, or DOM element** in the component's own template.

**`@ContentChild`** — Access content **projected into the component** via `<ng-content>`.

```typescript
// @ViewChild Example
import { Component, ViewChild, ElementRef, AfterViewInit } from '@angular/core';
import { ChartComponent } from './chart.component';

@Component({
  selector: 'app-dashboard',
  standalone: true,
  template: `
    <canvas #myCanvas></canvas>
    <app-chart #chartRef></app-chart>
    <button (click)="refreshChart()">Refresh</button>
  `
})
export class DashboardComponent implements AfterViewInit {
  @ViewChild('myCanvas') canvasRef!: ElementRef<HTMLCanvasElement>;
  @ViewChild(ChartComponent) chart!: ChartComponent;

  ngAfterViewInit() {
    // Safe to access here — view is fully rendered
    const ctx = this.canvasRef.nativeElement.getContext('2d');
    this.chart.initialize();
  }

  refreshChart() {
    this.chart.refresh();
  }
}
```

```typescript
// @ContentChild Example — Card component accepting projected content
import { Component, ContentChild, AfterContentInit } from '@angular/core';
import { CardHeaderComponent } from './card-header.component';

@Component({
  selector: 'app-card',
  standalone: true,
  template: `
    <div class="card">
      <ng-content select="app-card-header"></ng-content>
      <ng-content></ng-content>
    </div>
  `
})
export class CardComponent implements AfterContentInit {
  @ContentChild(CardHeaderComponent) header!: CardHeaderComponent;

  ngAfterContentInit() {
    // Safe to access here — content is projected
    console.log('Header text:', this.header.title);
  }
}

// Usage
// <app-card>
//   <app-card-header title="My Card"></app-card-header>
//   <p>Card body content</p>
// </app-card>
```

**`@ViewChildren` and `@ContentChildren`** — Get a `QueryList` of multiple matches:
```typescript
@ViewChildren(ChartComponent) charts!: QueryList<ChartComponent>;
ngAfterViewInit() {
  this.charts.forEach(chart => chart.initialize());
}
```

| Decorator | Scope | Access in |
|---|---|---|
| `@ViewChild` | Own template | `ngAfterViewInit` |
| `@ContentChild` | Projected content | `ngAfterContentInit` |
| `@ViewChildren` | All matching in own template | `ngAfterViewInit` |
| `@ContentChildren` | All matching in projected content | `ngAfterContentInit` |

---

## Q9. What is Change Detection in Angular? What is `OnPush` strategy?

**Answer:**
**Change Detection** is Angular's mechanism for syncing the component state (TypeScript) with the DOM (HTML template). Angular checks if data has changed and updates the view accordingly.

**Default strategy** — Angular checks every component in the tree on every event:
```
User clicks button → Zone.js detects async event →
Angular runs change detection on EVERY component from root →
Updates DOM where values changed
```

**`OnPush` strategy** — Angular only checks a component when:
1. An `@Input()` reference changes (not a mutated object)
2. An event originates from the component or its children
3. An Observable emits via `async` pipe
4. `ChangeDetectorRef.markForCheck()` is called manually

```typescript
import { Component, ChangeDetectionStrategy, Input } from '@angular/core';

@Component({
  selector: 'app-product-card',
  standalone: true,
  changeDetection: ChangeDetectionStrategy.OnPush,  // ← Key line
  template: `
    <div>{{ product.name }} — {{ product.price | currency }}</div>
  `
})
export class ProductCardComponent {
  @Input() product!: { name: string; price: number };
}
```

**OnPush + Signals (Angular 16+):** Signals automatically notify Angular when their value changes, making them the ideal companion for OnPush:
```typescript
@Component({
  changeDetection: ChangeDetectionStrategy.OnPush,
  template: `<p>{{ count() }}</p>`
})
export class CounterComponent {
  count = signal(0);
  increment() { this.count.update(v => v + 1); }
  // Signal change automatically triggers re-render
}
```

**Performance impact:**

| Strategy | Components checked per event | Use when |
|---|---|---|
| `Default` | All components | Simple apps, prototyping |
| `OnPush` | Only changed components | Large apps, performance-critical |

> **Trap:** With `OnPush`, mutating an object `this.items.push(item)` will NOT trigger re-render because the reference hasn't changed. Always create a new reference: `this.items = [...this.items, item]`

---

## Q10. What are Angular template syntax features? Explain `*ngIf`, `*ngFor`, `[ngClass]`, `[ngStyle]`.

**Answer:**
Angular templates use special syntax to bind data and control rendering.

**1. `*ngIf` — Conditional Rendering (NgModule era)**
```html
<!-- Adds/removes element from DOM -->
<div *ngIf="isLoggedIn">Welcome, {{ user.name }}</div>
<div *ngIf="isLoggedIn; else guestBlock">Welcome!</div>
<ng-template #guestBlock><p>Please log in</p></ng-template>

<!-- With async pipe -->
<div *ngIf="user$ | async as user">{{ user.name }}</div>
```

**Modern: `@if` (Angular 17+ built-in control flow):**
```html
@if (isLoggedIn) {
  <div>Welcome, {{ user.name }}</div>
} @else {
  <p>Please log in</p>
}
```

**2. `*ngFor` — List Rendering**
```html
<ul>
  <li *ngFor="let item of items; let i = index; trackBy: trackById">
    {{ i + 1 }}. {{ item.name }}
  </li>
</ul>
```

**Modern: `@for`:**
```html
<ul>
  @for (item of items; track item.id; let i = $index) {
    <li>{{ i + 1 }}. {{ item.name }}</li>
  } @empty {
    <li>No items found</li>
  }
</ul>
```

**3. `[ngClass]` — Dynamic CSS Classes**
```html
<!-- Object syntax -->
<div [ngClass]="{ 'active': isActive, 'disabled': isDisabled, 'premium': user.isPremium }">
  ...
</div>

<!-- Array syntax -->
<div [ngClass]="['card', isLarge ? 'card-lg' : 'card-sm']">...</div>

<!-- String syntax -->
<div [ngClass]="currentTheme">...</div>
```

**4. `[ngStyle]` — Dynamic Inline Styles**
```html
<div [ngStyle]="{
  'color': textColor,
  'font-size': fontSize + 'px',
  'background-color': isHighlighted ? 'yellow' : 'white'
}">
  Styled content
</div>
```

**Template syntax summary:**

| Syntax | Type | Example |
|---|---|---|
| `{{ value }}` | Interpolation | `{{ user.name }}` |
| `[property]` | Property binding | `[src]="imageUrl"` |
| `(event)` | Event binding | `(click)="onClick()"` |
| `[(ngModel)]` | Two-way binding | `[(ngModel)]="username"` |
| `#ref` | Template reference | `#myInput` → `myInput.value` |
| `*directive` | Structural directive | `*ngIf`, `*ngFor` |

> **Follow-up:** "What is `trackBy` in `*ngFor`?"
> `trackBy` tells Angular how to identify items in the list. Without it, Angular re-renders the entire list on any change. With it, Angular only re-renders items whose tracked value changed.
> ```typescript
> trackById(index: number, item: Product): string { return item.id; }
> ```

# Category 3 — Directives, Pipes & Data Binding

---

## Q11. What are Angular Directives? What is the difference between Structural and Attribute Directives?

**Answer:**
A **Directive** is a class that adds behavior to elements in Angular templates. There are three types:

| Type | Purpose | Examples |
|---|---|---|
| **Component** | Directive with a template | Every component is a directive |
| **Structural** | Change DOM layout — add/remove elements | `*ngIf`, `*ngFor`, `*ngSwitch`, `@if`, `@for` |
| **Attribute** | Change appearance or behavior of an element | `[ngClass]`, `[ngStyle]`, custom directives |

**Custom Attribute Directive:**
```typescript
// highlight.directive.ts — Changes background on hover
import { Directive, ElementRef, HostListener, Input } from '@angular/core';

@Directive({
  selector: '[appHighlight]',  // Attribute selector
  standalone: true,
})
export class HighlightDirective {
  @Input() appHighlight = 'yellow';  // Input with same name as selector
  @Input() defaultColor  = 'white';

  constructor(private el: ElementRef) {}

  @HostListener('mouseenter')
  onMouseEnter() {
    this.highlight(this.appHighlight);
  }

  @HostListener('mouseleave')
  onMouseLeave() {
    this.highlight(this.defaultColor);
  }

  private highlight(color: string) {
    this.el.nativeElement.style.backgroundColor = color;
  }
}
```

```html
<!-- Usage -->
<p appHighlight="lightblue" defaultColor="white">
  Hover over me!
</p>
```

**Custom Structural Directive:**
```typescript
// unless.directive.ts — Opposite of *ngIf
import { Directive, Input, TemplateRef, ViewContainerRef } from '@angular/core';

@Directive({
  selector: '[appUnless]',
  standalone: true,
})
export class UnlessDirective {
  private hasView = false;

  constructor(
    private templateRef: TemplateRef<any>,
    private viewContainer: ViewContainerRef
  ) {}

  @Input() set appUnless(condition: boolean) {
    if (!condition && !this.hasView) {
      this.viewContainer.createEmbeddedView(this.templateRef);
      this.hasView = true;
    } else if (condition && this.hasView) {
      this.viewContainer.clear();
      this.hasView = false;
    }
  }
}
```

```html
<p *appUnless="isLoggedIn">Please log in first.</p>
```

> **Trap:** Structural directives use `*` prefix as syntactic sugar. `*ngIf="condition"` expands to `<ng-template [ngIf]="condition"><...></ng-template>`.

---

## Q12. What are Angular Pipes? How do you create a custom pipe?

**Answer:**
**Pipes** transform data in templates without changing the underlying data. They use the `|` operator.

**Built-in pipes:**
```html
<!-- String pipes -->
{{ 'hello world' | uppercase }}           <!-- HELLO WORLD -->
{{ 'HELLO WORLD' | lowercase }}           <!-- hello world -->
{{ 'hello world' | titlecase }}           <!-- Hello World -->
{{ longText | slice:0:100 }}              <!-- First 100 chars -->

<!-- Number pipes -->
{{ 1234.567 | number:'1.2-2' }}          <!-- 1,234.57 -->
{{ 0.85 | percent }}                      <!-- 85% -->
{{ 99.99 | currency:'USD':'symbol':'1.2-2' }}  <!-- $99.99 -->

<!-- Date pipes -->
{{ today | date }}                        <!-- Sep 15, 2024 -->
{{ today | date:'dd/MM/yyyy' }}           <!-- 15/09/2024 -->
{{ today | date:'fullDate' }}             <!-- Sunday, September 15, 2024 -->

<!-- Other pipes -->
{{ data | json }}                         <!-- JSON stringify (debugging) -->
{{ observable$ | async }}                 <!-- Subscribe + unsubscribe automatically -->
{{ items | keyvalue }}                    <!-- Iterate over object keys/values -->
```

**Custom Pipe:**
```typescript
// truncate.pipe.ts
import { Pipe, PipeTransform } from '@angular/core';

@Pipe({
  name: 'truncate',
  standalone: true,
  pure: true,  // Default: only recalculates when input changes (reference)
})
export class TruncatePipe implements PipeTransform {
  transform(value: string, limit: number = 100, ellipsis: string = '...'): string {
    if (!value) return '';
    if (value.length <= limit) return value;
    return value.substring(0, limit) + ellipsis;
  }
}
```

```html
<!-- Usage -->
{{ article.body | truncate }}               <!-- truncate at 100 chars -->
{{ article.body | truncate:50 }}            <!-- truncate at 50 chars -->
{{ article.body | truncate:200:'… read more' }}

<!-- Chaining pipes -->
{{ article.title | truncate:30 | uppercase }}
```

**Pure vs Impure Pipes:**

| Type | `pure` | When recalculated | Use for |
|---|---|---|---|
| Pure | `true` (default) | Only when input reference changes | Stateless transforms (safe, fast) |
| Impure | `false` | Every change detection cycle | Array mutations, Date.now(), random values |

```typescript
@Pipe({ name: 'filterItems', pure: false })  // Impure — sees array mutations
export class FilterItemsPipe implements PipeTransform {
  transform(items: any[], filterText: string): any[] {
    return items.filter(item => item.name.includes(filterText));
  }
}
```

> **Trap:** Using an impure pipe on large arrays inside `*ngFor` causes severe performance issues — it runs on every change detection cycle. Prefer filtering in the component class instead.

---

## Q13. Explain all four types of Data Binding in Angular.

**Answer:**
Angular has four data binding mechanisms that connect the TypeScript class and the HTML template:

**1. Interpolation `{{ }}` — Component → Template (one-way)**
```html
<h1>{{ title }}</h1>
<p>2 + 2 = {{ 2 + 2 }}</p>
<img alt="{{ user.name }}'s avatar">
```

**2. Property Binding `[property]` — Component → Template (one-way)**
```html
<!-- Binds TypeScript expression to DOM property -->
<img [src]="user.avatarUrl" [alt]="user.name">
<button [disabled]="isLoading">Submit</button>
<input [value]="searchQuery">
<app-card [title]="product.name" [price]="product.price">

<!-- Attribute binding (for HTML attributes without DOM property equivalents) -->
<td [attr.colspan]="colSpan">...</td>
<button [attr.aria-label]="buttonLabel">...</button>

<!-- Class binding -->
<div [class.active]="isActive">...</div>
<div [class]="cssClass">...</div>

<!-- Style binding -->
<p [style.color]="textColor">...</p>
<p [style.font-size.px]="fontSize">...</p>
```

**3. Event Binding `(event)` — Template → Component (one-way)**
```html
<button (click)="onClick()">Click me</button>
<button (click)="onClick($event)">Click with event</button>
<input (keyup)="onKeyUp($event)" (blur)="onBlur()">
<input (keyup.enter)="onSearch()">   <!-- Key filter: fires only on Enter -->
<form (ngSubmit)="onSubmit()">...</form>

<!-- Custom component events -->
<app-product (addedToCart)="handleCartAdd($event)">
```

```typescript
onClick(event?: MouseEvent) {
  console.log('Clicked at:', event?.clientX, event?.clientY);
}
```

**4. Two-way Binding `[(ngModel)]` — Both directions**
```html
<!-- Requires FormsModule / ReactiveFormsModule -->
<input [(ngModel)]="username" placeholder="Enter name">
<p>Hello, {{ username }}</p>

<!-- [(ngModel)] is syntax sugar for: -->
<input [ngModel]="username" (ngModelChange)="username = $event">
```

**Custom two-way binding:**
```typescript
// counter.component.ts
@Component({ template: `...` })
export class CounterComponent {
  @Input()  value = 0;
  @Output() valueChange = new EventEmitter<number>();  // Must be: name + "Change"

  increment() { this.valueChange.emit(this.value + 1); }
}
```

```html
<!-- Parent can use two-way binding -->
<app-counter [(value)]="myCount"></app-counter>
```

**Summary table:**

| Syntax | Direction | Example |
|---|---|---|
| `{{ expr }}` | Component → DOM | `{{ title }}` |
| `[prop]` | Component → DOM | `[disabled]="true"` |
| `(event)` | DOM → Component | `(click)="fn()"` |
| `[(prop)]` | Both | `[(ngModel)]="name"` |

---

## Q14. What is `ngModel`? What is the difference between using it in Template-driven vs Reactive forms?

**Answer:**
`ngModel` is an Angular directive that creates a **two-way binding** between a form control in the template and a property in the component class.

**Template-driven — `ngModel` usage (simple forms):**
```typescript
// Requires FormsModule
import { FormsModule } from '@angular/forms';

@Component({
  standalone: true,
  imports: [FormsModule],
  template: `
    <form (ngSubmit)="onSubmit()" #myForm="ngForm">
      <input name="email" [(ngModel)]="user.email" required email #email="ngModel">
      <div *ngIf="email.invalid && email.touched">
        Email is required and must be valid
      </div>

      <input name="password" type="password" [(ngModel)]="user.password"
             required minlength="8" #pwd="ngModel">

      <button type="submit" [disabled]="myForm.invalid">Login</button>
    </form>
  `
})
export class LoginComponent {
  user = { email: '', password: '' };

  onSubmit() {
    console.log(this.user);
  }
}
```

**Reactive forms — programmatic (preferred for complex forms):**
```typescript
import { ReactiveFormsModule, FormBuilder, Validators } from '@angular/forms';

@Component({
  standalone: true,
  imports: [ReactiveFormsModule],
  template: `
    <form [formGroup]="loginForm" (ngSubmit)="onSubmit()">
      <input formControlName="email">
      <div *ngIf="email.invalid && email.touched">
        {{ email.errors?.['required'] ? 'Required' : 'Invalid email' }}
      </div>

      <input type="password" formControlName="password">

      <button [disabled]="loginForm.invalid">Login</button>
    </form>
  `
})
export class LoginComponent {
  loginForm = this.fb.group({
    email:    ['', [Validators.required, Validators.email]],
    password: ['', [Validators.required, Validators.minLength(8)]]
  });

  get email() { return this.loginForm.get('email')!; }

  constructor(private fb: FormBuilder) {}

  onSubmit() {
    if (this.loginForm.valid) console.log(this.loginForm.value);
  }
}
```

**Comparison:**

| Feature | Template-driven | Reactive |
|---|---|---|
| Setup | `FormsModule` | `ReactiveFormsModule` |
| Form definition | In template | In component class |
| Validation | HTML attributes | `Validators.*` functions |
| Testing | Harder (needs DOM) | Easier (pure class testing) |
| Dynamic forms | Difficult | Easy (`FormArray`) |
| Best for | Simple / small forms | Complex / dynamic forms |

---

## Q15. What is the `async` pipe? Why is it preferred over manual subscriptions in templates?

**Answer:**
The **`async` pipe** automatically **subscribes** to an Observable or Promise and **returns the latest emitted value**. It also **automatically unsubscribes** when the component is destroyed.

```typescript
@Component({
  standalone: true,
  imports: [AsyncPipe, NgIf],
  template: `
    <!-- async pipe handles subscribe + unsubscribe -->
    @if (user$ | async; as user) {
      <h1>{{ user.name }}</h1>
      <p>Email: {{ user.email }}</p>
    }

    <!-- With loading state -->
    @if (products$ | async; as products) {
      @for (product of products; track product.id) {
        <app-product-card [product]="product" />
      }
    } @else {
      <p>Loading products...</p>
    }
  `
})
export class UserDashboardComponent {
  user$     = this.userService.getCurrentUser();
  products$ = this.productService.getFeatured();

  constructor(
    private userService: UserService,
    private productService: ProductService
  ) {}
  // No ngOnDestroy needed — async pipe cleans up automatically
}
```

**Why NOT use manual subscriptions in templates:**
```typescript
// BAD — memory leak risk
@Component({ template: `<p>{{ user?.name }}</p>` })
export class BadComponent implements OnInit, OnDestroy {
  user: User | null = null;
  private sub!: Subscription;

  ngOnInit() {
    this.sub = this.userService.getUser().subscribe(u => this.user = u);
  }

  ngOnDestroy() {
    this.sub.unsubscribe();  // Easy to forget!
  }
}
```

```typescript
// GOOD — async pipe handles lifecycle
@Component({ template: `<p>{{ (user$ | async)?.name }}</p>` })
export class GoodComponent {
  user$ = this.userService.getUser();
  constructor(private userService: UserService) {}
}
```

**Advantages of `async` pipe:**

| Feature | Manual Subscribe | `async` Pipe |
|---|---|---|
| Auto-unsubscribe | ❌ Must do manually | ✅ Automatic |
| Memory leak risk | High | None |
| Triggers OnPush CD | ❌ Must call `markForCheck()` | ✅ Automatic |
| Code verbosity | More | Less |
| Multiple subscriptions | Complex | `@let x = obs$ \| async` |

> **Follow-up (Angular 16+):** Use `toSignal()` from `@angular/core/rxjs-interop` to convert Observables to Signals — even cleaner than `async` pipe:
> ```typescript
> user = toSignal(this.userService.getUser());
> // Template: {{ user()?.name }}
> ```

# Category 4 — Services, Dependency Injection & HTTP

---

## Q16. What is Dependency Injection (DI) in Angular? How does it work?

**Answer:**
**Dependency Injection** is a design pattern where a class receives its dependencies from an external source rather than creating them itself. Angular has a built-in hierarchical DI system that manages service instances.

**Without DI (tight coupling — bad):**
```typescript
export class OrderComponent {
  private orderService = new OrderService();   // Creates its own instance
  private httpClient   = new HttpClient();     // Impossible — requires Angular internals
}
```

**With DI (loose coupling — good):**
```typescript
@Component({ selector: 'app-order', standalone: true })
export class OrderComponent {
  // Angular's injector creates and provides these
  constructor(
    private orderService: OrderService,
    private authService: AuthService,
    private router: Router
  ) {}
}

// Modern Angular — inject() function (preferred)
@Component({ selector: 'app-order', standalone: true })
export class OrderComponent {
  private orderService = inject(OrderService);
  private authService  = inject(AuthService);
  private router       = inject(Router);
}
```

**Registering a service (3 ways):**
```typescript
// 1. providedIn: 'root' — Singleton, tree-shakeable (PREFERRED)
@Injectable({ providedIn: 'root' })
export class AuthService { }

// 2. Component-level — new instance per component
@Component({
  providers: [UserService]  // New instance for this component and its children
})
export class UserComponent { }

// 3. Module-level (NgModule era)
@NgModule({ providers: [AdminService] })
export class AdminModule { }
```

**DI Hierarchy:**
```
Root Injector (providedIn: 'root')
  └── Module Injector (NgModule providers)
        └── Component Injector (@Component providers)
              └── Child Component Injector
```

Angular walks **up** the injector hierarchy when resolving a dependency. The first matching provider wins.

**DI Decorators:**

| Decorator | Purpose |
|---|---|
| `@Injectable()` | Marks a class as injectable |
| `@Inject(TOKEN)` | Inject by token (for non-class values) |
| `@Optional()` | Don't throw if dependency not found |
| `@Self()` | Only look in current component's injector |
| `@SkipSelf()` | Skip current injector, start from parent |
| `@Host()` | Only look up to the host element injector |

---

## Q17. What is the difference between `providedIn: 'root'`, `'platform'`, and component-level providers?

**Answer:**

| Scope | Syntax | Instances | Lifetime |
|---|---|---|---|
| **Root** | `providedIn: 'root'` | 1 singleton for entire app | App lifetime |
| **Platform** | `providedIn: 'platform'` | 1 singleton across multiple Angular apps on page | Platform lifetime |
| **Any** | `providedIn: 'any'` | 1 instance per lazy-loaded module | Module lifetime |
| **Component** | `@Component({ providers: [...] })` | New instance per component | Component lifetime |
| **NgModule** | `@NgModule({ providers: [...] })` | 1 per module | Module lifetime |

```typescript
// Root singleton — shared across entire app
@Injectable({ providedIn: 'root' })
export class AuthService {
  private currentUser = signal<User | null>(null);
}

// Component-scoped — each instance of the component gets its own service
@Component({
  selector: 'app-form',
  providers: [FormStateService]  // Isolated state per form instance
})
export class FormComponent { }
```

**Tree-shaking:** Services with `providedIn: 'root'` that are never injected are **removed** from the production bundle by the build tool. Services in `NgModule.providers` are always included.

---

## Q18. What is `HttpClient`? How do you make GET, POST, PUT, DELETE requests?

**Answer:**
`HttpClient` is Angular's built-in HTTP service for making HTTP requests. It returns **Observables** (not Promises), enabling powerful RxJS operators.

**Setup:**
```typescript
// app.config.ts (standalone)
import { provideHttpClient, withInterceptors } from '@angular/common/http';

export const appConfig: ApplicationConfig = {
  providers: [
    provideHttpClient(withInterceptors([authInterceptor]))
  ]
};
```

**Service with all HTTP methods:**
```typescript
import { Injectable, inject } from '@angular/core';
import { HttpClient, HttpHeaders, HttpParams } from '@angular/common/http';
import { Observable } from 'rxjs';
import { map, catchError } from 'rxjs/operators';

export interface Product {
  id: number;
  name: string;
  price: number;
}

@Injectable({ providedIn: 'root' })
export class ProductService {
  private http = inject(HttpClient);
  private apiUrl = 'https://api.example.com/products';

  // GET — fetch all with query params
  getAll(category?: string): Observable<Product[]> {
    let params = new HttpParams();
    if (category) params = params.set('category', category);
    return this.http.get<Product[]>(this.apiUrl, { params });
  }

  // GET — fetch single by ID
  getById(id: number): Observable<Product> {
    return this.http.get<Product>(`${this.apiUrl}/${id}`);
  }

  // POST — create
  create(product: Omit<Product, 'id'>): Observable<Product> {
    const headers = new HttpHeaders({ 'Content-Type': 'application/json' });
    return this.http.post<Product>(this.apiUrl, product, { headers });
  }

  // PUT — full update
  update(id: number, product: Product): Observable<Product> {
    return this.http.put<Product>(`${this.apiUrl}/${id}`, product);
  }

  // PATCH — partial update
  partialUpdate(id: number, changes: Partial<Product>): Observable<Product> {
    return this.http.patch<Product>(`${this.apiUrl}/${id}`, changes);
  }

  // DELETE
  delete(id: number): Observable<void> {
    return this.http.delete<void>(`${this.apiUrl}/${id}`);
  }
}
```

**Component consuming the service:**
```typescript
@Component({
  standalone: true,
  imports: [AsyncPipe, NgFor],
  template: `
    @for (product of products$ | async; track product.id) {
      <app-product-card [product]="product" />
    }
  `
})
export class ProductListComponent {
  private productService = inject(ProductService);
  products$ = this.productService.getAll();
}
```

> **Follow-up:** "How do you handle errors in HttpClient?"
> ```typescript
> import { catchError, throwError } from 'rxjs';
>
> getAll(): Observable<Product[]> {
>   return this.http.get<Product[]>(this.apiUrl).pipe(
>     catchError(error => {
>       console.error('API Error:', error.status, error.message);
>       return throwError(() => new Error('Failed to load products'));
>     })
>   );
> }
> ```

---

## Q19. What is an HTTP Interceptor? How do you create one?

**Answer:**
An **HTTP Interceptor** is a service that intercepts every outgoing HTTP request or incoming response — allowing you to modify them centrally. Common uses: add auth headers, handle errors globally, log requests, show loading spinners.

**Functional Interceptor (Angular 15+, preferred):**
```typescript
// auth.interceptor.ts
import { HttpInterceptorFn, HttpRequest, HttpHandlerFn } from '@angular/common/http';
import { inject } from '@angular/core';
import { AuthService } from './auth.service';
import { catchError, throwError } from 'rxjs';
import { Router } from '@angular/router';

export const authInterceptor: HttpInterceptorFn = (
  req: HttpRequest<unknown>,
  next: HttpHandlerFn
) => {
  const authService = inject(AuthService);
  const router      = inject(Router);
  const token       = authService.getToken();

  // Clone request — HttpRequest is immutable
  const authReq = token
    ? req.clone({ headers: req.headers.set('Authorization', `Bearer ${token}`) })
    : req;

  return next(authReq).pipe(
    catchError(error => {
      if (error.status === 401) {
        authService.logout();
        router.navigate(['/login']);
      }
      if (error.status === 403) {
        router.navigate(['/forbidden']);
      }
      return throwError(() => error);
    })
  );
};
```

```typescript
// logging.interceptor.ts
export const loggingInterceptor: HttpInterceptorFn = (req, next) => {
  const start = Date.now();
  console.log(`→ ${req.method} ${req.url}`);

  return next(req).pipe(
    tap({
      next: () => console.log(`← ${req.url} (${Date.now() - start}ms)`),
      error: err => console.error(`✗ ${req.url} failed:`, err.message),
    })
  );
};
```

**Register interceptors:**
```typescript
// app.config.ts
import { provideHttpClient, withInterceptors } from '@angular/common/http';

export const appConfig: ApplicationConfig = {
  providers: [
    provideHttpClient(
      withInterceptors([
        loggingInterceptor,  // Applied first
        authInterceptor,     // Applied second
      ])
    )
  ]
};
```

**Interceptor order:** Interceptors are applied in the order they are registered. Responses flow in reverse order.

---

## Q20. What is RxJS? What are the most important operators used in Angular?

**Answer:**
**RxJS (Reactive Extensions for JavaScript)** is a library for composing asynchronous and event-based programs using **Observables**. Angular uses RxJS extensively — `HttpClient`, `Router`, `Forms`, and more all return Observables.

**Core concepts:**
```typescript
import { Observable, Subject, BehaviorSubject, of, from, interval } from 'rxjs';
import { map, filter, switchMap, mergeMap, concatMap, exhaustMap,
         debounceTime, distinctUntilChanged, takeUntil, catchError,
         tap, combineLatest, forkJoin } from 'rxjs/operators';
```

**Most important operators for Angular:**

| Operator | Purpose | Use Case |
|---|---|---|
| `map` | Transform each value | Transform API response data |
| `filter` | Filter values | Skip null/undefined values |
| `switchMap` | Cancel previous, switch to new Observable | Search autocomplete (cancel old requests) |
| `mergeMap` (flatMap) | Merge all Observables concurrently | Parallel HTTP requests |
| `concatMap` | Queue — process one at a time | Sequential operations |
| `exhaustMap` | Ignore new until current completes | Login button (ignore double clicks) |
| `debounceTime` | Wait for pause before emitting | Search box typing delay |
| `distinctUntilChanged` | Skip duplicate values | Avoid re-running search for same query |
| `takeUntil` | Complete when another Observable emits | Unsubscribe on component destroy |
| `catchError` | Handle errors | Global HTTP error handling |
| `tap` | Side effects without changing stream | Logging, loading state |
| `combineLatest` | Combine latest from multiple Observables | Dashboard with multiple data sources |
| `forkJoin` | Wait for all to complete | Load multiple APIs in parallel |

**Real-world examples:**
```typescript
// Search with debounce + switchMap (cancels stale requests)
searchResults$ = this.searchControl.valueChanges.pipe(
  debounceTime(300),
  distinctUntilChanged(),
  filter(query => query.length >= 2),
  switchMap(query => this.searchService.search(query)),
  catchError(err => of([]))
);

// Load user + orders in parallel
userWithOrders$ = forkJoin({
  user:   this.userService.getUser(id),
  orders: this.orderService.getOrdersForUser(id),
});

// Unsubscribe pattern with takeUntil
private destroy$ = new Subject<void>();

ngOnInit() {
  this.someStream$.pipe(
    takeUntil(this.destroy$)
  ).subscribe(data => this.data = data);
}

ngOnDestroy() {
  this.destroy$.next();
  this.destroy$.complete();
}
```

**Subject types:**

| Type | Behavior | Initial value |
|---|---|---|
| `Subject` | No replay — subscribers miss past values | None |
| `BehaviorSubject(val)` | Replays last value to new subscribers | Required |
| `ReplaySubject(n)` | Replays last N values | None |
| `AsyncSubject` | Emits only last value on completion | None |

# Category 5 — Routing & Navigation

---

## Q21. How does Angular Routing work? How do you set up routes?

**Answer:**
Angular's **Router** maps URL paths to components and manages navigation between views within the SPA — without full page reloads.

**Route setup (Standalone app):**
```typescript
// app.routes.ts
import { Routes } from '@angular/router';

export const routes: Routes = [
  { path: '',          component: HomeComponent },
  { path: 'products',  component: ProductListComponent },
  { path: 'products/:id', component: ProductDetailComponent },
  { path: 'about',     component: AboutComponent },
  { path: 'admin',
    loadChildren: () => import('./admin/admin.routes').then(m => m.adminRoutes)
  },
  { path: '**', component: NotFoundComponent }  // Wildcard — must be last
];
```

```typescript
// app.config.ts
import { provideRouter, withComponentInputBinding } from '@angular/router';

export const appConfig: ApplicationConfig = {
  providers: [
    provideRouter(routes, withComponentInputBinding())
  ]
};
```

```html
<!-- app.component.html — Router outlet renders matched component -->
<nav>
  <a routerLink="/">Home</a>
  <a routerLink="/products">Products</a>
  <a routerLink="/about">About</a>
</nav>

<router-outlet></router-outlet>
```

**Navigation methods:**
```html
<!-- Template navigation -->
<a routerLink="/products/42">View Product</a>
<a [routerLink]="['/products', product.id]">View Product</a>
<a routerLink="/products" [queryParams]="{ sort: 'price' }">Sort by Price</a>

<!-- Active link styling -->
<a routerLink="/products" routerLinkActive="active-link">Products</a>
<a routerLink="/" routerLinkActive="active" [routerLinkActiveOptions]="{ exact: true }">Home</a>
```

```typescript
// Programmatic navigation
import { Router, ActivatedRoute } from '@angular/router';

@Component({ ... })
export class ProductComponent {
  private router = inject(Router);
  private route  = inject(ActivatedRoute);

  goToDetail(id: string) {
    this.router.navigate(['/products', id]);
    // With query params:
    this.router.navigate(['/products'], { queryParams: { category: 'laptops' } });
    // Relative navigation:
    this.router.navigate(['../list'], { relativeTo: this.route });
  }
}
```

**Reading route parameters:**
```typescript
// Method 1: Snapshot (synchronous — for initial load)
const id = this.route.snapshot.paramMap.get('id');

// Method 2: Observable (for navigating between same component)
this.route.paramMap.subscribe(params => {
  const id = params.get('id');
});

// Method 3: Angular 16+ — Router Input Binding (withComponentInputBinding)
@Input() id!: string;  // Automatically bound from route param
```

---

## Q22. What is Lazy Loading? How do you implement it?

**Answer:**
**Lazy Loading** is a strategy where Angular only loads a module or component's JavaScript bundle **when the user navigates to that route**, rather than loading everything upfront. This significantly reduces the initial bundle size and improves application startup time.

**Without lazy loading (eagerly loaded):**
```typescript
// All routes and their components load at startup
const routes: Routes = [
  { path: 'admin', component: AdminDashboardComponent },  // Loaded always
];
```

**With lazy loading (standalone components — preferred):**
```typescript
// app.routes.ts
const routes: Routes = [
  { path: '', component: HomeComponent },  // Eagerly loaded

  // Lazy-loaded individual component
  {
    path: 'profile',
    loadComponent: () => import('./profile/profile.component')
      .then(m => m.ProfileComponent)
  },

  // Lazy-loaded feature route group
  {
    path: 'admin',
    loadChildren: () => import('./admin/admin.routes')
      .then(m => m.adminRoutes)
  },

  // Lazy-loaded with guard
  {
    path: 'dashboard',
    canActivate: [authGuard],
    loadComponent: () => import('./dashboard/dashboard.component')
      .then(m => m.DashboardComponent)
  }
];
```

```typescript
// admin/admin.routes.ts — lazy-loaded route module
export const adminRoutes: Routes = [
  { path: '',       component: AdminHomeComponent },
  { path: 'users',  component: AdminUsersComponent },
  { path: 'reports',component: AdminReportsComponent },
];
```

**How Angular handles it:**
1. User navigates to `/admin`
2. Angular router checks if `admin.js` chunk is loaded
3. If not → fetches `admin.[hash].js` from server
4. Once loaded → renders `AdminHomeComponent`

**Preloading strategy — load lazy modules in background after app boots:**
```typescript
import { provideRouter, withPreloading, PreloadAllModules } from '@angular/router';

provideRouter(routes, withPreloading(PreloadAllModules))
```

**Performance impact example:**
- Without lazy loading: Initial bundle 2.4MB → 8.2s TTI
- With lazy loading: Initial bundle 380KB → 1.4s TTI

---

## Q23. What are Route Guards? What types exist?

**Answer:**
**Route Guards** are interfaces that control whether a route can be activated, deactivated, or its children accessed. They return `true` (allow), `false` (deny + navigate away), or a `UrlTree` (redirect).

**Guard types:**

| Guard | Interface | Purpose |
|---|---|---|
| `CanActivate` | `CanActivateFn` | Can user enter this route? |
| `CanActivateChild` | `CanActivateChildFn` | Can user enter child routes? |
| `CanDeactivate` | `CanDeactivateFn<T>` | Can user leave this route? (unsaved changes) |
| `CanMatch` | `CanMatchFn` | Should this route match at all? (feature flags) |
| `Resolve` | `ResolveFn<T>` | Pre-load data before component activates |

**Functional Guards (Angular 15+, preferred):**
```typescript
// auth.guard.ts
import { inject } from '@angular/core';
import { CanActivateFn, Router } from '@angular/router';
import { AuthService } from './auth.service';
import { map } from 'rxjs';

export const authGuard: CanActivateFn = (route, state) => {
  const authService = inject(AuthService);
  const router      = inject(Router);

  if (authService.isLoggedIn()) return true;

  // Redirect to login with return URL
  return router.createUrlTree(['/login'], {
    queryParams: { returnUrl: state.url }
  });
};

// Role-based guard
export const adminGuard: CanActivateFn = () => {
  const authService = inject(AuthService);
  const router      = inject(Router);

  return authService.hasRole('admin')
    ? true
    : router.createUrlTree(['/forbidden']);
};

// CanDeactivate — prevent losing unsaved form data
export const unsavedChangesGuard: CanDeactivateFn<{ hasUnsavedChanges(): boolean }> =
  (component) => {
    if (component.hasUnsavedChanges()) {
      return confirm('You have unsaved changes. Leave anyway?');
    }
    return true;
  };
```

```typescript
// routes with guards
const routes: Routes = [
  {
    path: 'dashboard',
    component: DashboardComponent,
    canActivate: [authGuard],
    canDeactivate: [unsavedChangesGuard],
  },
  {
    path: 'admin',
    canActivate: [authGuard, adminGuard],
    children: [
      { path: 'users', component: AdminUsersComponent },
    ]
  }
];
```

---

## Q24. What is a Route Resolver? When should you use it?

**Answer:**
A **Resolver** is a guard that **pre-fetches data before the component is activated**. The component only renders after the data is ready — eliminating loading spinners inside the component.

```typescript
// product.resolver.ts
import { ResolveFn } from '@angular/router';
import { inject } from '@angular/core';
import { ProductService } from './product.service';
import { Product } from './product.model';

export const productResolver: ResolveFn<Product> = (route) => {
  const productService = inject(ProductService);
  const id = route.paramMap.get('id')!;
  return productService.getById(id);  // HTTP call resolves before component loads
};
```

```typescript
// Routes with resolver
const routes: Routes = [
  {
    path: 'products/:id',
    component: ProductDetailComponent,
    resolve: { product: productResolver }  // Data key name
  }
];
```

```typescript
// Component receives pre-loaded data
@Component({ ... })
export class ProductDetailComponent implements OnInit {
  private route = inject(ActivatedRoute);
  product!: Product;

  ngOnInit() {
    // Data is already loaded — no async handling needed
    this.product = this.route.snapshot.data['product'];
  }
}
```

**When to use Resolver vs component-level loading:**

| Approach | Resolver | Component `ngOnInit` |
|---|---|---|
| Component rendered | After data ready | Immediately |
| Loading spinner | In router transition | Inside component |
| URL changes | After data loaded | Immediately |
| Best for | SEO, avoiding flicker | Interactive dashboards |

> **Trap:** Resolvers block navigation — if the API call is slow, the user sees a blank/frozen page. Always add a timeout or error handling.

---

## Q25. What is `ActivatedRoute`? How do you read query params and route params?

**Answer:**
`ActivatedRoute` provides information about the **currently active route** — path params, query params, fragment, and resolved data.

```typescript
@Component({
  selector: 'app-product-list',
  standalone: true,
  template: `
    @for (product of products$ | async; track product.id) {
      <app-product-card [product]="product" />
    }
  `
})
export class ProductListComponent implements OnInit {
  private route         = inject(ActivatedRoute);
  private productService = inject(ProductService);

  products$!: Observable<Product[]>;

  ngOnInit() {
    // React to query param changes (e.g., ?category=laptops&sort=price)
    this.products$ = this.route.queryParamMap.pipe(
      map(params => ({
        category: params.get('category') ?? 'all',
        sort:     params.get('sort')     ?? 'name',
        page:     Number(params.get('page') ?? 1),
      })),
      switchMap(filters => this.productService.getAll(filters))
    );
  }
}

@Component({ selector: 'app-product-detail', standalone: true })
export class ProductDetailComponent implements OnInit {
  private route         = inject(ActivatedRoute);
  private productService = inject(ProductService);
  product$!: Observable<Product>;

  ngOnInit() {
    // React to route param changes (e.g., /products/:id)
    this.product$ = this.route.paramMap.pipe(
      map(params => params.get('id')!),
      switchMap(id => this.productService.getById(id))
    );
  }
}
```

**All ActivatedRoute properties:**

| Property | Type | Description |
|---|---|---|
| `snapshot` | `ActivatedRouteSnapshot` | Current state (synchronous, one-time) |
| `paramMap` | `Observable<ParamMap>` | Route params (reactive) |
| `queryParamMap` | `Observable<ParamMap>` | Query params (reactive) |
| `data` | `Observable<Data>` | Resolver + static data |
| `fragment` | `Observable<string>` | URL fragment (#section) |
| `url` | `Observable<UrlSegment[]>` | URL segments |
| `parent` | `ActivatedRoute` | Parent route |

```typescript
// Snapshot (use when param won't change while component is alive)
const id       = this.route.snapshot.paramMap.get('id');
const category = this.route.snapshot.queryParamMap.get('category');
const product  = this.route.snapshot.data['product'];  // from resolver
```

# Category 6 — Forms (Q26–Q30)

---

## Q26. What is the difference between Template-driven and Reactive Forms?

**Answer:**
Angular provides two approaches to building forms:

| Feature | Template-driven | Reactive |
|---|---|---|
| Module | `FormsModule` | `ReactiveFormsModule` |
| Form model defined in | Template (HTML) | Component class (TypeScript) |
| Form control access | Template refs (`#f="ngForm"`) | `FormControl`, `FormGroup` objects |
| Validation | HTML attributes (`required`, `minlength`) | `Validators.*` functions |
| Dynamic forms | Difficult | Easy (`FormArray`) |
| Unit testing | Requires DOM/TestBed | Pure class testing |
| Async validation | Supported | Fully supported |
| Observability | Limited | `valueChanges`, `statusChanges` Observables |
| Best for | Simple / contact forms | Complex / wizard / dynamic forms |

**Template-driven form:**
```typescript
@Component({
  standalone: true,
  imports: [FormsModule, NgIf],
  template: `
    <form #f="ngForm" (ngSubmit)="submit(f)">
      <input name="name" ngModel required minlength="2" #name="ngModel">
      <div *ngIf="name.invalid && name.touched">Name is required (min 2 chars)</div>

      <input name="email" ngModel required email #email="ngModel">
      <div *ngIf="email.errors?.['email']">Must be a valid email</div>

      <button [disabled]="f.invalid">Submit</button>
    </form>
  `
})
export class ContactFormComponent {
  submit(form: NgForm) {
    if (form.valid) console.log(form.value); // { name: '...', email: '...' }
  }
}
```

**Reactive form:**
```typescript
@Component({
  standalone: true,
  imports: [ReactiveFormsModule, NgIf],
  template: `
    <form [formGroup]="form" (ngSubmit)="submit()">
      <input formControlName="name">
      <div *ngIf="name.invalid && name.touched">{{ getNameError() }}</div>

      <input formControlName="email">
      <button [disabled]="form.invalid">Submit</button>
    </form>
  `
})
export class ContactReactiveComponent {
  private fb = inject(FormBuilder);

  form = this.fb.group({
    name:  ['', [Validators.required, Validators.minLength(2)]],
    email: ['', [Validators.required, Validators.email]],
  });

  get name() { return this.form.get('name')!; }

  getNameError() {
    if (this.name.errors?.['required'])   return 'Name is required';
    if (this.name.errors?.['minlength'])  return 'Name must be at least 2 characters';
    return '';
  }

  submit() {
    if (this.form.valid) console.log(this.form.value);
  }
}
```

---

## Q27. What are `FormControl`, `FormGroup`, and `FormArray`?

**Answer:**
These are the three building blocks of Angular Reactive Forms:

**`FormControl`** — tracks the value and validity of a single form field:
```typescript
import { FormControl, Validators } from '@angular/forms';

const emailControl = new FormControl('', {
  validators: [Validators.required, Validators.email],
  nonNullable: true,  // Angular 14+ — value is never null
});

emailControl.value;          // ''
emailControl.valid;          // false
emailControl.errors;         // { required: true }
emailControl.setValue('a@b.com');
emailControl.patchValue('x@y.com');
emailControl.valueChanges.subscribe(v => console.log(v));  // Observable
emailControl.statusChanges.subscribe(s => console.log(s)); // 'VALID' | 'INVALID'
```

**`FormGroup`** — groups multiple `FormControl`s:
```typescript
const profileForm = new FormGroup({
  firstName: new FormControl('', Validators.required),
  lastName:  new FormControl('', Validators.required),
  address: new FormGroup({
    street: new FormControl(''),
    city:   new FormControl(''),
    zip:    new FormControl('', Validators.pattern(/^\d{5}$/)),
  })
});

profileForm.value;                         // { firstName: '', lastName: '', address: {...} }
profileForm.get('address.city')?.value;    // Nested access
profileForm.patchValue({ firstName: 'John' }); // Partial update
profileForm.setValue({ firstName: 'John', lastName: 'Doe', address: {...} }); // Full update
profileForm.reset();                       // Reset to initial values
```

**`FormArray`** — array of `FormControl`s or `FormGroup`s (dynamic lists):
```typescript
const form = this.fb.group({
  title: ['', Validators.required],
  tags:  this.fb.array(['Angular', 'TypeScript']),  // FormArray
  skills: this.fb.array([
    this.fb.group({ name: [''], level: ['beginner'] })
  ])
});

// Access FormArray
get tags() { return this.form.get('tags') as FormArray; }
get skills() { return this.form.get('skills') as FormArray; }

// Add item
addTag() { this.tags.push(this.fb.control('', Validators.required)); }

// Remove item
removeTag(index: number) { this.tags.removeAt(index); }
```

```html
<!-- FormArray in template -->
<div formArrayName="tags">
  @for (tag of tags.controls; track $index; let i = $index) {
    <input [formControlName]="i" placeholder="Tag {{ i + 1 }}">
    <button type="button" (click)="removeTag(i)">✕</button>
  }
  <button type="button" (click)="addTag()">+ Add Tag</button>
</div>
```

---

## Q28. How do you create custom validators in Angular?

**Answer:**
Custom validators are functions that take a `AbstractControl` and return a validation error object or `null` (valid).

**Synchronous custom validator:**
```typescript
import { AbstractControl, ValidationErrors, ValidatorFn } from '@angular/forms';

// Standalone validator function
export function noSpacesValidator(): ValidatorFn {
  return (control: AbstractControl): ValidationErrors | null => {
    if (!control.value) return null;  // Skip if empty (let required handle that)
    const hasSpaces = /\s/.test(control.value);
    return hasSpaces ? { noSpaces: { value: control.value } } : null;
  };
}

// Password strength validator
export function passwordStrengthValidator(): ValidatorFn {
  return (control: AbstractControl): ValidationErrors | null => {
    const value: string = control.value || '';
    const errors: ValidationErrors = {};

    if (!/[A-Z]/.test(value))  errors['missingUppercase'] = true;
    if (!/[a-z]/.test(value))  errors['missingLowercase'] = true;
    if (!/[0-9]/.test(value))  errors['missingNumber']    = true;
    if (!/[^A-Za-z0-9]/.test(value)) errors['missingSpecial'] = true;

    return Object.keys(errors).length ? errors : null;
  };
}

// Cross-field validator — passwords must match (attached to FormGroup)
export function passwordMatchValidator(): ValidatorFn {
  return (group: AbstractControl): ValidationErrors | null => {
    const password        = group.get('password')?.value;
    const confirmPassword = group.get('confirmPassword')?.value;
    return password === confirmPassword ? null : { passwordMismatch: true };
  };
}
```

```typescript
// Usage in Reactive Form
form = this.fb.group({
  username:        ['', [Validators.required, noSpacesValidator()]],
  password:        ['', [Validators.required, passwordStrengthValidator()]],
  confirmPassword: ['', Validators.required],
}, { validators: passwordMatchValidator() });
```

**Async custom validator (e.g., username availability check):**
```typescript
import { AsyncValidatorFn, AbstractControl, ValidationErrors } from '@angular/forms';
import { Observable, of } from 'rxjs';
import { map, catchError, debounceTime, switchMap, first } from 'rxjs/operators';

export function usernameAvailableValidator(userService: UserService): AsyncValidatorFn {
  return (control: AbstractControl): Observable<ValidationErrors | null> => {
    if (!control.value) return of(null);

    return of(control.value).pipe(
      debounceTime(400),
      switchMap(username => userService.checkAvailability(username)),
      map(isAvailable => isAvailable ? null : { usernameTaken: true }),
      catchError(() => of(null)),
      first()  // Complete after first emission
    );
  };
}
```

```typescript
// Usage — async validators go in third argument
username = new FormControl('', {
  validators: [Validators.required, noSpacesValidator()],
  asyncValidators: [usernameAvailableValidator(this.userService)],
  updateOn: 'blur'  // Validate on blur instead of every keystroke
});
```

---

## Q29. What are the states of a Form Control (`valid`, `invalid`, `touched`, `dirty`)?

**Answer:**
Every `FormControl`, `FormGroup`, and `FormArray` tracks both **value state** and **interaction state**:

**Value state:**

| Property | Meaning |
|---|---|
| `valid` | All validators pass |
| `invalid` | At least one validator fails |
| `pending` | Async validator is running |
| `disabled` | Control is disabled (excluded from value/validation) |
| `errors` | Object of validation errors, or `null` if valid |
| `value` | Current value |

**Interaction state:**

| Property | Opposite | Set when |
|---|---|---|
| `pristine` | `dirty` | User has NOT yet changed the value |
| `dirty` | `pristine` | User HAS changed the value |
| `untouched` | `touched` | User has NOT yet blurred the field |
| `touched` | `untouched` | User HAS blurred (focused then unfocused) the field |

**Best practice — only show validation errors after the user has interacted:**
```html
<!-- Show error only when: invalid AND user has interacted (touched) -->
<input formControlName="email" (blur)="emailControl.markAsTouched()">

@if (email.invalid && email.touched) {
  @if (email.errors?.['required']) {
    <span class="error">Email is required</span>
  }
  @if (email.errors?.['email']) {
    <span class="error">Must be a valid email address</span>
  }
}
```

**Programmatic control of state:**
```typescript
control.markAsTouched();          // Mark as touched (show validation)
control.markAsDirty();            // Mark as dirty
control.markAsPristine();         // Reset to pristine
control.markAsUntouched();        // Reset to untouched
control.disable();                // Disable (excluded from form value)
control.enable();                 // Enable
control.setErrors({ custom: true }); // Set errors manually
control.clearValidators();        // Remove validators
control.addValidators(Validators.required);
control.updateValueAndValidity(); // Re-run validation
```

**CSS classes Angular applies automatically:**

| State | CSS class added |
|---|---|
| `valid` | `.ng-valid` |
| `invalid` | `.ng-invalid` |
| `touched` | `.ng-touched` |
| `untouched` | `.ng-untouched` |
| `dirty` | `.ng-dirty` |
| `pristine` | `.ng-pristine` |

```css
/* Highlight invalid touched fields */
input.ng-invalid.ng-touched { border-color: red; }
input.ng-valid.ng-touched   { border-color: green; }
```

---

## Q30. What are common Angular performance optimization techniques?

**Answer:**
Angular performance optimization involves multiple layers — from build configuration to runtime rendering:

**1. OnPush Change Detection Strategy**
```typescript
@Component({
  changeDetection: ChangeDetectionStrategy.OnPush
})
// Reduces change detection checks to only when @Input reference changes
```

**2. Lazy Loading Routes**
```typescript
{ path: 'admin', loadChildren: () => import('./admin/admin.routes').then(m => m.routes) }
// Initial bundle size: 2.4MB → 380KB
```

**3. Lazy Loading Images**
```html
<!-- HTML native lazy loading (Angular 15+ NgOptimizedImage) -->
<img ngSrc="hero.jpg" width="800" height="600" priority>
<img ngSrc="below-fold.jpg" width="400" height="300">  <!-- Auto lazy loaded -->
```

**4. `trackBy` in `*ngFor`**
```typescript
trackById(index: number, item: Product) { return item.id; }
// Prevents re-rendering unchanged items in a list
```

**5. Unsubscribe from Observables**
```typescript
// Prevent memory leaks — 3 approaches:
// a) async pipe (automatic)
// b) takeUntilDestroyed() (Angular 16+)
stream$.pipe(takeUntilDestroyed()).subscribe(...);
// c) takeUntil + Subject
```

**6. Pure Pipes over Component Methods**
```html
<!-- BAD: Called on every change detection cycle -->
{{ getFormattedPrice(product.price) }}

<!-- GOOD: Pure pipe — only recalculates when input changes -->
{{ product.price | currency:'USD' }}
```

**7. Signals (Angular 16+)**
```typescript
// Signals enable targeted, granular re-renders
count = signal(0);
// Only components reading count() re-render when count changes
```

**8. `@defer` Blocks (Angular 17+)**
```html
<!-- Load heavy components only when needed -->
@defer (on viewport) {
  <app-heavy-chart />
}
```

**9. Production Build + Bundle Analysis**
```bash
ng build --configuration production
npx webpack-bundle-analyzer dist/my-app/stats.json
```

**10. Server-Side Rendering (SSR)**
```bash
ng add @angular/ssr
# Improves FCP and LCP — critical content rendered on server
```

**Performance checklist summary:**

| Optimization | Impact Level | Effort |
|---|---|---|
| Production build | High | Low |
| Lazy loading routes | High | Low |
| OnPush strategy | High | Medium |
| `trackBy` in lists | Medium | Low |
| `async` pipe (no manual subscribe) | Medium | Low |
| Pure pipes | Medium | Low |
| Signals (Angular 16+) | High | Medium |
| `@defer` blocks (Angular 17+) | High | Low |
| `NgOptimizedImage` | Medium | Low |
| SSR + Hydration | High | High |

# Quick Reference Cheatsheet — Set 1

---

## Angular Core Concepts at a Glance

### Component Anatomy
```typescript
@Component({
  selector: 'app-example',
  standalone: true,
  imports: [CommonModule, RouterLink],    // Direct imports (standalone)
  templateUrl: './example.component.html',
  styleUrl:    './example.component.scss',
  changeDetection: ChangeDetectionStrategy.OnPush,
})
export class ExampleComponent implements OnInit, OnDestroy {
  // Signal inputs (Angular 16+)
  title    = input.required<string>();
  maxItems = input(10);

  // Signal outputs (Angular 18+)
  itemSelected = output<string>();

  // Local signals
  count = signal(0);
  total = computed(() => this.count() * 2);

  // Injected services
  private router = inject(Router);

  ngOnInit()    { /* Initialize, fetch data */ }
  ngOnDestroy() { /* Cleanup subscriptions */ }
}
```

---

### Lifecycle Hooks — Order & Purpose

```
1. ngOnChanges    → @Input changed
2. ngOnInit       → Component ready → FETCH DATA HERE
3. ngDoCheck      → Every CD cycle (avoid)
4. ngAfterContentInit    → <ng-content> projected
5. ngAfterContentChecked → After content check
6. ngAfterViewInit       → View ready → ACCESS ViewChild HERE
7. ngAfterViewChecked    → After view check
8. ngOnDestroy           → Cleanup → UNSUBSCRIBE HERE
```

---

### Data Binding Quick Reference

```html
{{ expression }}          ← Interpolation  (Component → DOM)
[property]="expr"         ← Property binding (Component → DOM)
(event)="handler($event)" ← Event binding (DOM → Component)
[(ngModel)]="field"       ← Two-way binding (Both directions)
[attr.aria-label]="expr"  ← Attribute binding
[class.active]="bool"     ← Class binding
[style.color]="expr"      ← Style binding
#ref                      ← Template reference variable
```

---

### Dependency Injection Quick Reference

```typescript
// Best: providedIn root (singleton + tree-shakeable)
@Injectable({ providedIn: 'root' })
export class MyService {}

// inject() function (preferred over constructor injection)
private myService = inject(MyService);

// Component-scoped service (new instance per component)
@Component({ providers: [MyService] })
```

---

### RxJS Operators Cheatsheet

```typescript
// Transform
.pipe(map(v => v * 2))

// Filter
.pipe(filter(v => v > 0))

// HTTP with cancellation (search)
.pipe(switchMap(q => this.http.get(`/search?q=${q}`)))

// HTTP parallel (no cancellation)
.pipe(mergeMap(id => this.http.get(`/item/${id}`)))

// HTTP sequential
.pipe(concatMap(item => this.http.post('/save', item)))

// Ignore new clicks while processing (login button)
.pipe(exhaustMap(creds => this.auth.login(creds)))

// Search box
.pipe(debounceTime(300), distinctUntilChanged())

// Unsubscribe (Angular 16+)
.pipe(takeUntilDestroyed())

// Error handling
.pipe(catchError(err => of(defaultValue)))

// Combine latest from multiple streams
combineLatest([user$, orders$]).pipe(map(([user, orders]) => ...))

// Wait for all to complete
forkJoin({ user: user$, orders: orders$ })
```

---

### Forms Quick Reference

```typescript
// Reactive form setup
form = this.fb.group({
  name:  ['', [Validators.required, Validators.minLength(2)]],
  email: ['', [Validators.required, Validators.email]],
  tags:  this.fb.array([]),    // Dynamic list
});

// Access controls
get name()  { return this.form.get('name')!; }
get tags()  { return this.form.get('tags') as FormArray; }

// Built-in validators
Validators.required
Validators.email
Validators.minLength(n)
Validators.maxLength(n)
Validators.min(n)
Validators.max(n)
Validators.pattern(/regex/)
```

---

### Routing Quick Reference

```typescript
// Route definition
{ path: 'products/:id', component: ProductDetailComponent,
  canActivate: [authGuard],
  resolve: { product: productResolver }
}

// Lazy loading
{ path: 'admin', loadChildren: () => import('./admin/admin.routes').then(m => m.routes) }
{ path: 'profile', loadComponent: () => import('./profile.component').then(m => m.ProfileComponent) }

// Programmatic navigation
this.router.navigate(['/products', id]);
this.router.navigate(['/products'], { queryParams: { sort: 'price' } });

// Reading params
this.route.paramMap.subscribe(p => p.get('id'));
this.route.snapshot.queryParamMap.get('sort');
```

---

### Key Interview Topics Summary

| Topic | Q# | Key Point |
|---|---|---|
| Angular vs AngularJS | Q1 | TypeScript, component-based, 6-month release cycle |
| NgModule | Q2 | declarations, imports, exports, providers |
| Standalone | Q3 | `standalone: true`, imports in component, Angular 19 default |
| CLI | Q4 | `ng g`, `ng build`, `ng serve`, `ng update` |
| Build process | Q5 | AOT, esbuild, tree-shaking, dist output |
| Lifecycle hooks | Q6 | 8 hooks, init=fetch data, afterViewInit=ViewChild, destroy=unsubscribe |
| @Input/@Output | Q7 | Property vs event binding, EventEmitter / output() |
| ViewChild/ContentChild | Q8 | Own template vs projected content |
| Change detection | Q9 | Default vs OnPush, signals, markForCheck |
| Template syntax | Q10 | `*ngIf`, `*ngFor`, `@if`, `@for`, ngClass, ngStyle |
| Directives | Q11 | Structural vs Attribute, HostListener, ElementRef |
| Pipes | Q12 | Pure vs Impure, PipeTransform, built-in pipes |
| Data binding | Q13 | 4 types: interpolation, property, event, two-way |
| ngModel | Q14 | Template-driven vs Reactive forms |
| async pipe | Q15 | Auto subscribe/unsubscribe, OnPush compatible |
| DI | Q16 | Hierarchical injector, @Injectable, inject() |
| Providers | Q17 | providedIn root vs component level, tree-shaking |
| HttpClient | Q18 | Returns Observable, GET/POST/PUT/DELETE, HttpParams |
| Interceptors | Q19 | Functional interceptors, clone request, error handling |
| RxJS | Q20 | switchMap/mergeMap/exhaustMap/concatMap, Subject types |
| Routing | Q21 | Routes array, router-outlet, routerLink, navigate() |
| Lazy loading | Q22 | loadComponent/loadChildren, PreloadAllModules |
| Guards | Q23 | CanActivate, CanDeactivate, CanMatch — functional style |
| Resolvers | Q24 | Pre-load data before route activates |
| ActivatedRoute | Q25 | paramMap, queryParamMap, snapshot, data |
| Template vs Reactive | Q26 | FormsModule vs ReactiveFormsModule, complexity tradeoffs |
| FormControl/Group/Array | Q27 | Reactive form building blocks, FormArray for dynamic lists |
| Custom validators | Q28 | ValidatorFn, AsyncValidatorFn, cross-field validators |
| Control states | Q29 | valid/invalid, touched/untouched, dirty/pristine |
| Performance | Q30 | OnPush, lazy loading, trackBy, async pipe, signals, @defer |

---

> **Set 2 Preview:** State management (NgRx, Signals Store), Testing (TestBed, Spectator, Jest), SSR & Hydration, Angular Signals deep dive, Micro-frontends, Angular CDK, Accessibility.